# Part 2: blinded human rating (P2-EXP-001)

**Run only the code cell below** (kernel: the Docker Jupyter server, *Python 3 (ipykernel)*). It loads no model and uses no GPU.

1. Click **your own** rater button (Rater 1, 2 or 3). Rate independently and don't discuss the images until all three raters have finished.
2. For each pair H01–H25 you see the complete original JPG (prompt printed at the top, LEFT and RIGHT images). Answer both questions with **LEFT** or **RIGHT** (no ties):
   - **Pokémon-style appearance**: which image looks more like the Pokémon artwork style of the training data?
   - **Prompt matching**: which image matches the prompt text better?
3. Press **Save & Next**. It is enabled only when both answers are chosen, and it saves the pair immediately.
   **Back** goes to your previous pair to correct it. **Pause** returns to the start screen.
4. To resume, even after closing VS Code or restarting the kernel, run the cell again and click your rater button. You continue at your first unfinished pair.
5. After Rater 3 finishes, the start screen enables **Export final 150-row CSV**, which writes `implementation/eval/p2_human_ratings.csv` after validation.
   Then run `p2_finetune.ipynb` with `MODE = "REPORT_ONLY"`.

Responses: `/app/data266_lab2_work/p2/human_eval/ui_progress/RATER_n_progress.csv` → `rater_n_completed.csv` (outside Git).
The answer key is never opened by this notebook.


In [1]:
# Part 2 blinded human rating (P2-EXP-001). RUN THIS CELL. No model, no GPU, no generation.
# Reads only rater_sheet.csv, p2_human_ratings_TEMPLATE.csv and rater_packet/H01..H25.jpg; never opens answer_key_PRIVATE.json.
import csv, html, os, tempfile, traceback
from pathlib import Path
import ipywidgets as W
from IPython.display import display

PACKET_DIR = Path("/app/data266_lab2_work/p2/human_eval/P2-EXP-001/c8d369176621")
OUT_DIR = Path(os.environ.get("P2_RATING_OUT_DIR", "/app/data266_lab2_work/p2/human_eval"))         # env override: tests only
FINAL_PATH = Path(os.environ.get("P2_RATING_FINAL_PATH", "/app/implementation/eval/p2_human_ratings.csv"))
PROGRESS_DIR = OUT_DIR / "ui_progress"            # same per-rater files the earlier UI wrote (existing answers are kept)
RATERS = ("RATER_1", "RATER_2", "RATER_3")
CRITERIA = ("domain_style", "prompt_match")
COLUMNS = ["rater", "pair_id", "criterion", "choice"]
CHOICES = ("left", "right")
QUESTIONS = {"domain_style": "1. Pokémon-style appearance: which image looks more like the Pokémon artwork style of the training data?",
             "prompt_match": "2. Prompt matching: which image matches the prompt text better?"}

# ---------- packet (read-only) ----------
with open(PACKET_DIR / "rater_sheet.csv", newline="", encoding="utf-8") as f:
    SHEET = list(csv.DictReader(f))
PAIR_IDS = [r["pair_id"] for r in SHEET]
assert PAIR_IDS == [f"H{i:02d}" for i in range(1, 26)], PAIR_IDS
PROMPT = {r["pair_id"]: r["prompt"] for r in SHEET}
ITEMS = [(p, c) for p in PAIR_IDS for c in CRITERIA]
with open(PACKET_DIR / "p2_human_ratings_TEMPLATE.csv", newline="", encoding="utf-8") as f:
    _tmpl = list(csv.DictReader(f))
assert list(_tmpl[0]) == COLUMNS and [(r["pair_id"], r["criterion"]) for r in _tmpl if r["rater"] == "RATER_1"] == ITEMS
JPG = {}
for r in SHEET:
    path = PACKET_DIR / r["image_file"]
    assert path.parent.name == "rater_packet" and path.name == f"{r['pair_id']}.jpg", path
    JPG[r["pair_id"]] = path.read_bytes()          # original bytes, displayed unmodified
    assert JPG[r["pair_id"]][:2] == b"\xff\xd8", path

# ---------- storage ----------
def label(rid): return "Rater " + rid.split("_")[1]
def progress_path(rid): return PROGRESS_DIR / f"{rid}_progress.csv"
def completed_path(rid): return OUT_DIR / f"rater_{rid.split('_')[1]}_completed.csv"

def _write_atomic(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix="." + path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=COLUMNS, lineterminator="\n"); w.writeheader(); w.writerows(rows)
            f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp): os.unlink(tmp)

def _read(path):
    with open(path, newline="", encoding="utf-8") as f:
        return [dict(r) for r in csv.DictReader(f)]

def load(rid):
    ans = {it: "" for it in ITEMS}
    if progress_path(rid).exists():
        for r in _read(progress_path(rid)):
            key = (r["pair_id"], r["criterion"])
            if r["rater"] != rid or key not in ans or r["choice"] not in ("",) + CHOICES:
                raise RuntimeError(f"unexpected row in {progress_path(rid)}: {r}")
            ans[key] = r["choice"]
    return ans

def rows_of(rid, ans): return [{"rater": rid, "pair_id": p, "criterion": c, "choice": ans[(p, c)]} for p, c in ITEMS]

def check_rows(rows, raters):
    """Same rules as load_human_ratings() in p2_finetune.ipynb plus exact coverage per rater."""
    probs, need = [], set(ITEMS)
    for i, r in enumerate(rows):
        if list(r) != COLUMNS: probs.append(f"row {i}: columns {list(r)}"); continue
        if any(not (v or "").strip() for v in r.values()): probs.append(f"row {i}: empty field")
        if r["choice"] not in CHOICES: probs.append(f"row {i}: invalid choice {r['choice']!r}")
        if r["criterion"] not in CRITERIA: probs.append(f"row {i}: invalid criterion {r['criterion']!r}")
        if r["pair_id"] not in PAIR_IDS: probs.append(f"row {i}: unknown pair {r['pair_id']!r}")
        if r["rater"] not in raters: probs.append(f"row {i}: unexpected rater {r['rater']!r}")
    keys = [(r.get("rater"), r.get("pair_id"), r.get("criterion")) for r in rows]
    if len(keys) != len(set(keys)): probs.append(f"{len(keys) - len(set(keys))} duplicated (rater, pair_id, criterion) rows")
    for rid in raters:
        mine = [r for r in rows if r.get("rater") == rid]
        if len(mine) != 50 or {(r["pair_id"], r["criterion"]) for r in mine} != need:
            probs.append(f"{rid}: {len(mine)} rows / incomplete coverage (need 25 pairs x 2 criteria = 50)")
    if len(rows) != 50 * len(raters): probs.append(f"{len(rows)} rows in total, expected {50 * len(raters)}")
    return probs

def status(rid): return sum(bool(v) for v in load(rid).values()), completed_path(rid).exists()
def first_open_pair(rid):
    ans = load(rid)
    return next((i for i, p in enumerate(PAIR_IDS) if not all(ans[(p, c)] for c in CRITERIA)), len(PAIR_IDS) - 1)

def save_pair(rid, pid, sel):
    """Saves BOTH answers of one pair. Returns True when this rater now has all 50 (completed file written)."""
    if completed_path(rid).exists(): raise RuntimeError(f"{label(rid)} is completed and locked.")
    assert rid in RATERS and pid in PAIR_IDS and all(sel[c] in CHOICES for c in CRITERIA), (rid, pid, sel)
    ans = load(rid)
    for c in CRITERIA: ans[(pid, c)] = sel[c]
    rows = rows_of(rid, ans)
    _write_atomic(progress_path(rid), rows)
    if all(ans.values()):
        probs = check_rows(rows, (rid,))
        if probs: raise RuntimeError(probs)
        _write_atomic(completed_path(rid), rows)
        return True
    return False

def export_final():
    missing = [label(r) for r in RATERS if not completed_path(r).exists()]
    if missing: return False, "Not exported. Still incomplete: " + ", ".join(missing)
    rows = [r for rid in RATERS for r in _read(completed_path(rid))]
    probs = check_rows(rows, RATERS)
    if probs: return False, "Not exported, validation failed: " + "; ".join(probs[:10])
    fd, tmp = tempfile.mkstemp(dir=FINAL_PATH.parent, prefix=".p2_export.", suffix=".tmp"); os.close(fd); tmp = Path(tmp)
    try:
        _write_atomic(tmp, rows)
        if FINAL_PATH.exists():
            if FINAL_PATH.read_bytes() == tmp.read_bytes(): return True, f"Already exported: {FINAL_PATH} is identical (150 rows)."
            return False, f"Not exported: {FINAL_PATH} already exists with different content (left untouched)."
        os.replace(tmp, FINAL_PATH)
    finally:
        tmp.unlink(missing_ok=True)
    return True, f"Exported 150 rows (3 raters x 25 pairs x 2 criteria) to {FINAL_PATH}. Validated; no results computed."

# ---------- widgets ----------
S = {"rid": None, "pair": 0, "sel": {c: None for c in CRITERIA}}
BIG = W.Layout(width="240px", height="56px")
def big_button(text, **kw):
    b = W.Button(description=text, layout=kw.pop("layout", BIG), **kw); b.style.font_size = "18px"; b.style.font_weight = "bold"; return b

err = W.HTML()
home_info = W.HTML()
rater_btns = {r: big_button(label(r)) for r in RATERS}
export_btn = W.Button(description="Export final 150-row CSV", layout=W.Layout(width="300px", height="44px"))
export_msg = W.HTML()
home = W.VBox([W.HTML("<h2>Part 2: blinded image comparison</h2><p>Select <b>your own</b> rater number. Rate independently; "
                      "do not discuss the images until all three raters have finished. No ties: always choose LEFT or RIGHT.</p>"),
               W.HBox(list(rater_btns.values())), home_info, export_btn, export_msg])

header = W.HTML()
prompt_box = W.HTML()
image = W.Image(format="jpeg", layout=W.Layout(width="100%", max_width="1054px"))
pick = {(c, s): big_button(s.upper()) for c in CRITERIA for s in CHOICES}
q_rows = [W.VBox([W.HTML(f"<div style='font-size:17px;font-weight:600;margin-top:8px'>{html.escape(QUESTIONS[c])}</div>"),
                  W.HBox([pick[(c, "left")], pick[(c, "right")]])]) for c in CRITERIA]
note = W.HTML()
back_btn = W.Button(description="◀ Back", layout=W.Layout(width="140px", height="44px"))
save_btn = big_button("Save & Next ▶", button_style="primary")
jump_btn = W.Button(description="Jump to first unanswered", layout=W.Layout(width="220px", height="44px"))
pause_btn = W.Button(description="Pause", layout=W.Layout(width="140px", height="44px"))
rate = W.VBox([header, prompt_box, image, *q_rows, note, W.HBox([back_btn, save_btn, jump_btn, pause_btn])])

done_html = W.HTML()
done_back = W.Button(description="◀ Back (review my last pair)", layout=W.Layout(width="260px", height="44px"))
done_next = big_button("Start next rater", layout=W.Layout(width="260px", height="56px"), button_style="success")
done_home = W.Button(description="Start screen", layout=W.Layout(width="160px", height="44px"))
done = W.VBox([done_html, W.HBox([done_back, done_next, done_home])])
root = W.VBox([err, home])

def safe(fn):
    def h(*a):
        try:
            err.value = ""; fn(*a)
        except Exception:
            err.value = "<pre style='color:#b00020;white-space:pre-wrap'>" + html.escape(traceback.format_exc()) + "</pre>"
    return h

def show_home(msg=""):
    S["rid"] = None
    n_done = 0
    for r, b in rater_btns.items():
        n, comp = status(r); n_done += comp
        b.description = f"{label(r)}: completed ✓" if comp else (f"{label(r)}: resume ({n}/50)" if n else f"{label(r)}: start")
        b.disabled = comp; b.button_style = "" if comp else "info"
    home_info.value = f"<p><b>{n_done}/3 raters completed.</b> Completed raters are locked.</p>"
    export_btn.disabled = n_done < 3
    export_msg.value = msg
    root.children = [err, home]

def refresh_controls():
    for (c, s), b in pick.items():
        on = S["sel"][c] == s
        b.button_style = "success" if on else ""
        b.description = ("✓ " if on else "") + s.upper()
    save_btn.disabled = not all(S["sel"][c] in CHOICES for c in CRITERIA)

def show_pair(i):
    rid = S["rid"]; pid = PAIR_IDS[i]; S["pair"] = i
    ans = load(rid)
    S["sel"] = {c: (ans[(pid, c)] or None) for c in CRITERIA}          # this rater's OWN saved answers only
    n = sum(bool(v) for v in ans.values())
    header.value = (f"<div style='font-size:18px'><b>{label(rid)}</b> &nbsp;·&nbsp; Pair <b>{pid}/25</b> &nbsp;·&nbsp; "
                    f"<b>{n}/50</b> responses saved</div>"
                    f"<div style='height:8px;background:#ddd;border-radius:4px;margin:4px 0'><div style='height:8px;width:{2 * n}%;"
                    f"background:#2f9e44;border-radius:4px'></div></div>")
    prompt_box.value = (f"<div style='font-size:18px;margin:4px 0'><b>Prompt</b> (verbatim from rater_sheet.csv, "
                        f"also printed at the top of the image): <i>{html.escape(PROMPT[pid])}</i></div>")
    image.value = JPG[pid]
    saved = [c for c in CRITERIA if S["sel"][c]]
    note.value = ("<div style='color:#8a5a00;margin-top:6px'>Your earlier saved answer for this pair is pre-selected "
                  f"({', '.join(saved)}). Keep it or change it, then press Save &amp; Next.</div>") if saved else ""
    back_btn.disabled = i == 0
    refresh_controls()
    root.children = [err, rate]

def show_done(rid):
    nxt = next((r for r in RATERS if not completed_path(r).exists()), None)
    done_html.value = (f"<h2>{label(rid)} completed: 50/50 responses saved.</h2><p>Saved to <code>{completed_path(rid)}</code>. "
                       "Please do not discuss your answers with the other raters.</p>")
    done_next.description = f"Start {label(nxt)} ▶" if nxt else "All done: start screen"
    root.children = [err, done]

@safe
def on_rater(b):
    rid = next(r for r, x in rater_btns.items() if x is b)
    if completed_path(rid).exists(): return show_home(f"{label(rid)} is completed and locked.")
    S["rid"] = rid; show_pair(first_open_pair(rid))

def make_pick(c, s):
    @safe
    def h(_):
        if S["rid"]: S["sel"][c] = s; refresh_controls()
    return h

@safe
def on_save(_):
    if not S["rid"] or not all(S["sel"][c] in CHOICES for c in CRITERIA): return
    save_btn.disabled = True                                            # no double save
    rid, i = S["rid"], S["pair"]
    if save_pair(rid, PAIR_IDS[i], dict(S["sel"])): return show_done(rid)
    show_pair(i + 1 if i + 1 < len(PAIR_IDS) else first_open_pair(rid))

@safe
def on_back(_):
    if S["rid"] and S["pair"] > 0: show_pair(S["pair"] - 1)

@safe
def on_jump(_):
    if S["rid"]: show_pair(first_open_pair(S["rid"]))

@safe
def on_done_back(_):
    rid = S["rid"]
    if rid: completed_path(rid).unlink(missing_ok=True); show_pair(len(PAIR_IDS) - 1)

@safe
def on_done_next(_):
    nxt = next((r for r in RATERS if not completed_path(r).exists()), None)
    if nxt: S["rid"] = nxt; show_pair(first_open_pair(nxt))
    else: show_home()

@safe
def on_export(_):
    ok, msg = export_final()
    show_home(("✅ " if ok else "❌ ") + html.escape(msg))

for b in rater_btns.values(): b.on_click(on_rater)
for (c, s), b in pick.items(): b.on_click(make_pick(c, s))
save_btn.on_click(on_save); back_btn.on_click(on_back); jump_btn.on_click(on_jump)
pause_btn.on_click(safe(lambda _: show_home("Paused. Saved pairs are kept; press your rater button to resume.")))
done_back.on_click(on_done_back); done_next.on_click(on_done_next); done_home.on_click(safe(lambda _: show_home()))
export_btn.on_click(on_export)
show_home()
display(root)
